# Concatenate Indoor Walk Test CSV Files

This notebook combines every `.CSV` file in the `CSV` folder into `Concat_Indoor_Walk_Test_from_csv.csv` without dropping any measurement rows.

In [ ]:
from __future__ import annotations

import csv
from itertools import islice
from pathlib import Path

ROOT = Path.cwd()
CSV_DIR = ROOT / 'CSV'
OUTPUT_FILE = ROOT / 'Concat_Indoor_Walk_Test_from_csv.csv'

if not CSV_DIR.exists():
    raise FileNotFoundError(f'Missing input folder: {CSV_DIR}')

# Every scan type (Blind Scan, Top N Signal, etc.) exports its own CSV with a
# different column set, so we glob everything and reconcile columns below.
csv_files = sorted(CSV_DIR.glob('*.CSV'))
if not csv_files:
    raise FileNotFoundError('No .CSV files found in CSV/')

print(f'csv_files={len(csv_files)}')
print(f'output_file={OUTPUT_FILE}')

In [ ]:
# Pass 1: union every file's column headers (in first-seen order) and count
# rows, without holding all the data in memory at once.
fieldnames: list[str] = []
fieldname_set: set[str] = set()
total_rows = 0

for csv_file in csv_files:
    with csv_file.open('r', encoding='utf-8', errors='replace', newline='') as source:
        reader = csv.DictReader(source)
        if reader.fieldnames is None:
            continue

        for fieldname in reader.fieldnames:
            if fieldname not in fieldname_set:
                fieldname_set.add(fieldname)
                fieldnames.append(fieldname)

        for _ in reader:
            total_rows += 1

# Pass 2: stream every row into the merged file against the unioned header.
# extrasaction='ignore' + row.get(..., '') pads/truncates each row to the
# common schema so files missing a column don't break the writer.
with OUTPUT_FILE.open('w', encoding='utf-8', newline='') as destination:
    writer = csv.DictWriter(destination, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()

    written_rows = 0
    for csv_file in csv_files:
        with csv_file.open('r', encoding='utf-8', errors='replace', newline='') as source:
            reader = csv.DictReader(source)
            if reader.fieldnames is None:
                continue

            for row in reader:
                writer.writerow({fieldname: row.get(fieldname, '') for fieldname in fieldnames})
                written_rows += 1

print(f'columns={len(fieldnames)}')
print(f'input_rows={total_rows}')
print(f'output_rows={written_rows}')

# Sanity check: the two-pass counts must match, or rows were silently dropped.
if written_rows != total_rows:
    raise RuntimeError('Output row count does not match input row count')

In [3]:
with OUTPUT_FILE.open('r', encoding='utf-8', errors='replace', newline='') as merged_file:
    reader = csv.DictReader(merged_file)
    preview_rows = list(islice(reader, 5))

print(f'preview_rows={len(preview_rows)}')
for index, row in enumerate(preview_rows, start=1):
    print(f'Row {index}:')
    print({key: value for key, value in row.items() if value})
    print()

preview_rows=5
Row 1:
{'Date': '07/07/2026', 'Time': '14:03:08:144', 'Latitude': '38.903461', 'Longitude': '-77.007583', 'Loop Count': '8', 'Sweep Number': '1', 'Channel Number': '350', 'Channel Style': '6', 'Bandwidth': '20 MHz', 'Block Status': '0', 'Antenna Gain': '0', 'Cable Loss': '0', 'Channel RSSI': '-78.4', 'Adjusted Channel RSSI': '-78.4', 'Cell Id': '496', 'DSS': '1', 'Ref Signal - Timeoffset': '287127', 'Ref Signal - Timeoffset (msec)': '9.35', 'Ref Signal - Delay Spread': '5', 'Ref Signal - Received Power': '-101.27', 'Adjusted Ref Signal - Received Power': '-101.27', 'Ref Signal - Received Quality': '-7.79', 'Ref Signal - CINR': '12.97', 'Sync Signal - Timeoffset': '287118', 'Sync Signal - Timeoffset (msec)': '9.35', 'Primary Sync Channel Received Power': '-103.56', 'Adjusted Primary Sync Channel Received Power': '-103.56', 'Primary Sync Channel Received Quality': '-0.24', 'Secondary Sync Channel Received Power': '-103.56', 'Secondary Sync Channel Received Quality': '-0.49